<a href="https://colab.research.google.com/github/sanjivinicarmel/AgriDataAnalysis2016/blob/main/Fine_Tuning_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

We are NOT training BERT from scratch. We take a pre-trained DistilBERT model and continue training it on labelled movie reviews so that it learns the Positive/Negative classification task.

What we are going to do :

Pre-trained DistilBERT
        -
Already understands language
        -
Give it labelled movie reviews
        -
"Excellent movie!" → Positive
"Very boring movie" → Negative
        -
Model adjusts its weights
        -
Fine-tuned Sentiment Model

Steps that will be followed:

IMDb Dataset
     -
Train / Test Split
     -
Tokenization
     -
Pre-trained DistilBERT
     -
Fine-Tuning
     -
Evaluation
     -
New Review → Positive / Negative

In [ ]:
!pip install -q transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00


We need:

datasets → get IMDb dataset

transformers → BERT/DistilBERT and tokenizer

evaluate → calculate accuracy

accelerate → helps Hugging Face run training efficiently

In [ ]:
#Import Libraries
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,#Loads a pretrained Transformer model designed for classification tasks.
    TrainingArguments,#Defines how the model should be trained.
    Trainer#Handles the actual training and evaluation process.
)
import evaluate
import numpy as np

In [ ]:
!pip install -U datasets huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.5.1
    Uninstalling hf-xet-1.5.1:
      Successfully uninstalled hf-xet-1.5.1
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [ ]:
print(dataset["train"][0])

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

0--- negative
1----positive

In [ ]:
#Taking a small dataset for demo
train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(500))

In [ ]:
print(train_dataset)
print(test_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 2000
})
Dataset({
    features: ['text', 'label'],
    num_rows: 500
})


In [ ]:
#Loading Pre0trained DistilBert
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


This is a pre-trained Transformer model.

It has already learned general language patterns.

We are adding:
Sentiment Classification

        ↓

Negative / Positive
The num_labels=2 tells the model:

Oour classification problem has two classes.

DistilBERT is a smaller version of BERT designed to retain much of BERT's language capability with less computational cost.

In [ ]:
#Model overview before fine tuning
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer
)

In [ ]:
classifier("I absolutely loved this movie!")

[{'label': 'LABEL_1', 'score': 0.5221525430679321}]

In [ ]:
classifier("This movie was extremely boring.")

[{'label': 'LABEL_1', 'score': 0.5021417140960693}]

Because this model has just been loaded with a classification head that hasn't been trained for our IMDb labels, don't expect meaningful sentiment predictions yet.

In [ ]:
#Tokenization
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [ ]:
#Applying
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
#Inspecting
print(tokenized_train[0])

{'text': 'There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier\'s plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it\'s the language, or the spirit, but I think this series is more English than American. By the way, the actors are really good and funny. The acting is not superficial at all...', 'label': 1, 'input_ids': [101, 2045, 2003, 2053, 7189, 2012, 2035, 2090, 3481, 3771, 1998, 6337, 2099, 2021, 1996, 2755, 2008, 2119, 2024, 2610, 2186, 2055, 6355, 6997, 1012, 6337, 2099, 3504, 15594, 2100, 1010, 3481, 3771, 350

These numbers are called:

Token IDs / Input IDs

The Transformer does not directly receive English sentences.

It receives numerical representations.

Hugging Face notes that tokenization creates model inputs such as input_ids and attention_mask
https://huggingface.co/docs/transformers/training

In [ ]:
#We do not need the original text columns
tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_test = tokenized_test.remove_columns(["text"])

In [ ]:
print(tokenized_train.column_names)

['label', 'input_ids', 'token_type_ids', 'attention_mask']


In [ ]:
#checking accuracy
accuracy = evaluate.load("accuracy")

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    predictions = np.argmax(predictions, axis=1)

    return accuracy.compute(
        predictions=predictions,
        references=labels
    )

The model produces scores for:

Negative → 2.1
Positive → 4.8

We choose the class with the highest score.

So:

np.argmax(...)

means:

"Give the class with the highest score."


In [ ]:
#This is where fine tuning starts
training_args = TrainingArguments(
    output_dir="./sentiment_model",
    eval_strategy="epoch",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)

In [ ]:
#Creating Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
#starts fine tuning process
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.483431,0.442675,0.798000
2,0.285272,0.412338,0.824000


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=250, training_loss=0.41084755706787107, metrics={'train_runtime': 3064.6911, 'train_samples_per_second': 1.305, 'train_steps_per_second': 0.082, 'total_flos': 132467398656000.0, 'train_loss': 0.41084755706787107, 'epoch': 2.0})



Conceptually:

Movie Review
     ↓
Tokenizer
     ↓
Input IDs
     ↓
DistilBERT
     ↓
Prediction
     ↓
Compare with actual label
     ↓
Calculate Loss
     ↓
Backpropagation
     ↓
Update Weights
     ↓
Next Batch

For example:

Training example
Review:
"This movie was fantastic!"

Actual label:
1 → Positive

Model initially predicts:

Negative = 0.60
Positive = 0.40

That's wrong.

So the model calculates a loss.

Then:

Loss
 ↓
Backpropagation
 ↓
Gradients
 ↓
Weights updated

After seeing many examples, the model gradually becomes better at this task.

Hugging Face's Trainer handles batching, the forward pass, loss computation, backpropagation and parameter updates for you.
https://huggingface.co/docs/transformers/en/trainer

In [ ]:
#Evaluate Fine tuned model
results = trainer.evaluate()

print(results)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
0.285272,0.412338,2,0.824000


{'eval_loss': 0.41233760118484497, 'eval_accuracy': 0.824}


In [ ]:
#Savimng  the Fine-Tuned Model
trainer.save_model("./sentiment_model")
tokenizer.save_pretrained("./sentiment_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./sentiment_model/tokenizer_config.json', './sentiment_model/tokenizer.json')

Now we have:

sentiment_model/
       ↓
Fine-tuned DistilBERT

This is different from the original:

distilbert-base-uncased

We have adapted it to our specific task.

In [ ]:
#Use the Fine-Tuned Model
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="./sentiment_model",
    tokenizer="./sentiment_model"
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
#Testing
sentiment_model("This movie was absolutely fantastic!")

[{'label': 'LABEL_1', 'score': 0.9150597453117371}]

In [ ]:
sentiment_model("I hated this movie. It was very boring.")

[{'label': 'LABEL_0', 'score': 0.9070644378662109}]

In [ ]:
sentiment_model("The acting was excellent and the story was amazing.")

[{'label': 'LABEL_1', 'score': 0.9311306476593018}]

Label 0 = negative
Label 1 - positive

In [ ]:
#Extra info
while True:

    review = input("Enter a movie review (or type 'quit'): ")

    if review.lower() == "quit":
        break

    result = sentiment_model(review)

    print(result)

Enter a movie review (or type 'quit'): The movie was fantastic
[{'label': 'LABEL_1', 'score': 0.9087485671043396}]
Enter a movie review (or type 'quit'): quit


As we have fine tuned the distalled model the student can now use this model for sentiment analysis .